In [2]:
import json
import random
import re
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from sklearn.metrics import accuracy_score, classification_report
from sklearn.model_selection import train_test_split
from torch import nn
from torch.nn.utils.rnn import pack_padded_sequence
from torch.utils.data import DataLoader, Dataset
from transformers import BertTokenizer

seed = 42
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(seed)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Set up Hugging Face cache directory
cache_dir = Path(r"C:/Users/User/Documents/devanasokan_fyp/huggingface_cache")
cache_dir.mkdir(parents=True, exist_ok=True)
tokenizer = BertTokenizer.from_pretrained("bert-base-uncased", cache_dir=str(cache_dir))
print(f"Tokenizer vocab size: {len(tokenizer)}")

Using device: cuda
Tokenizer vocab size: 30522


In [3]:
data_path = Path(r"C:/Users/User/Documents/devanasokan_fyp/preparation/modeldata.csv")
df = pd.read_csv(data_path)
df = df[["verse", "label"]].dropna().drop_duplicates().reset_index(drop=True)

print(df.shape)
print(df["label"].value_counts().sort_index())

(22878, 2)
label
0    11439
1    11439
Name: count, dtype: int64


#### Text Cleaning and Train/Validation/Test Split

The RNN should only see text from the training split when building the vocabulary. That avoids leaking information from validation or test lyrics into the tokenizer.

In [4]:
def clean_text(text: str) -> str:
    text = str(text).lower()
    text = re.sub(r"[^a-z0-9\s']", " ", text) # Replace non-alphanumeric characters with spaces
    text = re.sub(r"\s+", " ", text).strip() # Replace multiple spaces with a single space and trim
    return text

train_df, temp_df = train_test_split(
    df,
    test_size=0.2,
    random_state=seed,
    stratify=df["label"],
)
val_df, test_df = train_test_split(
    temp_df,
    test_size=0.5,
    random_state=seed,
    stratify=temp_df["label"],
)

for frame in (train_df, val_df, test_df):
    frame.loc[:, "clean_verse"] = frame["verse"].apply(clean_text)

print("train:", train_df.shape, train_df["label"].value_counts().sort_index().to_dict())
print("val:", val_df.shape, val_df["label"].value_counts().sort_index().to_dict())
print("test:", test_df.shape, test_df["label"].value_counts().sort_index().to_dict())

train: (18302, 3) {0: 9151, 1: 9151}
val: (2288, 3) {0: 1144, 1: 1144}
test: (2288, 3) {0: 1144, 1: 1144}


In [5]:
MAX_LEN = 180


def tokenize(text: str) -> list[str]:
    return tokenizer.tokenize(text)


counter = Counter()
for text in train_df["clean_verse"]:
    counter.update(tokenize(text))

vocab = tokenizer.get_vocab()

print(f"Vocabulary size: {len(vocab)}")
print("Most common tokens:", counter.most_common(10))

Token indices sequence length is longer than the specified maximum sequence length for this model (544 > 512). Running this sequence through the model will result in indexing errors


Vocabulary size: 30522
Most common tokens: [('i', 54906), ('you', 40629), ('the', 31056), ('to', 24273), ('it', 20227), ('a', 19852), ('me', 18337), ('and', 17799), ('not', 15771), ('is', 15142)]


In [6]:
def numericalize(text: str) -> tuple[torch.Tensor, torch.Tensor]:
    token_ids = tokenizer.encode(
        text,
        add_special_tokens=False,
        truncation=True,
        max_length=MAX_LEN,
    )
    if not token_ids:
        token_ids = [tokenizer.unk_token_id]
    length = len(token_ids)
    if len(token_ids) < MAX_LEN:
        token_ids += [tokenizer.pad_token_id] * (MAX_LEN - len(token_ids))
    return torch.tensor(token_ids, dtype=torch.long), torch.tensor(length, dtype=torch.long)

sample_ids, sample_length = numericalize(train_df.iloc[0]["clean_verse"] )
print("sample length:", sample_length.item())
print("sample ids:", sample_ids[:20].tolist())

sample length: 46
sample ids: [2026, 2026, 2092, 2009, 2003, 25085, 2066, 2057, 2024, 2035, 2183, 2000, 3280, 2035, 2183, 2000, 3280, 9061, 9061, 2524]


In [7]:
class LyricsDataset(Dataset):
    def __init__(self, frame: pd.DataFrame):
        self.texts = frame["clean_verse"].tolist()
        self.labels = frame["label"].astype(np.float32).tolist()

    def __len__(self) -> int:
        return len(self.texts)

    def __getitem__(self, index: int):
        input_ids, length = numericalize(self.texts[index])
        label = torch.tensor(self.labels[index], dtype=torch.float32)
        return input_ids, length, label

## Hyperparameter Tuning with Optuna

We'll search for the best hyperparameters using Optuna, which efficiently explores different combinations and focuses on promising regions. We'll tune:
- Learning rate (1e-4 to 1e-2)
- Embedding dimension (64, 128, 256)
- Hidden dimension (128, 256, 512)
- Number of layers (1, 2, 3)
- Dropout rate (0.1 to 0.5)
- Batch size (16, 32, 64)

In [8]:
class RNNClassifier(nn.Module):
    def __init__(
        self,
        vocab_size: int,
        embedding_dim: int = 128,
        hidden_dim: int = 128,
        num_layers: int = 2,
        bidirectional: bool = True,
        dropout: float = 0.3,
    ):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=tokenizer.pad_token_id)
        self.rnn = nn.RNN(
            embedding_dim,
            hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            nonlinearity="tanh", # For RNN
            dropout=dropout if num_layers > 1 else 0.0,
            bidirectional=bidirectional,
        )
        output_dim = hidden_dim * 2 if bidirectional else hidden_dim
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(output_dim, 1)

    def forward(self, input_ids: torch.Tensor, lengths: torch.Tensor) -> torch.Tensor:
        embedded = self.embedding(input_ids)
        packed = pack_padded_sequence(
            embedded,
            lengths.cpu(),
            batch_first=True,
            enforce_sorted=False,
        )
        _, hidden = self.rnn(packed)
        if self.rnn.bidirectional:
            hidden = torch.cat((hidden[-2], hidden[-1]), dim=1)
        else:
            hidden = hidden[-1]
        logits = self.fc(self.dropout(hidden))
        return logits.squeeze(1)

In [9]:
def run_epoch(model, loader: DataLoader, criterion, optimizer=None, training: bool = True):
    model.train() if training else model.eval()

    total_loss = 0.0
    all_predictions = []
    all_targets = []

    for input_ids, lengths, labels in loader:
        input_ids = input_ids.to(device)
        lengths = lengths.to(device)
        labels = labels.to(device)

        with torch.set_grad_enabled(training):
            logits = model(input_ids, lengths)
            loss = criterion(logits, labels)

            if training:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

        predictions = (torch.sigmoid(logits) >= 0.5).long()
        all_predictions.extend(predictions.detach().cpu().tolist())
        all_targets.extend(labels.detach().cpu().long().tolist())
        total_loss += loss.item() * input_ids.size(0)

    average_loss = total_loss / len(loader.dataset)
    accuracy = accuracy_score(all_targets, all_predictions)
    return average_loss, accuracy

In [10]:
import optuna
from optuna.pruners import MedianPruner


def objective(trial: optuna.Trial):
    # Suggest hyperparameters
    learning_rate = trial.suggest_float("learning_rate", 1e-4, 1e-2, log=True)
    embedding_dim = trial.suggest_categorical("embedding_dim", [64, 128, 256])
    hidden_dim = trial.suggest_categorical("hidden_dim", [128, 256, 512])
    num_layers = trial.suggest_int("num_layers", 1, 3)
    dropout = trial.suggest_float("dropout", 0.1, 0.5)
    batch_size = trial.suggest_categorical("batch_size", [16, 32, 64])

    # Create data loaders with the suggested batch size
    train_loader = DataLoader(LyricsDataset(train_df), batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(LyricsDataset(val_df), batch_size=batch_size, shuffle=False)

    # Initialize model with suggested hyperparameters
    model = RNNClassifier(
        vocab_size=len(tokenizer),
        embedding_dim=embedding_dim,
        hidden_dim=hidden_dim,
        num_layers=num_layers,
        dropout=dropout,
    ).to(device)

    criterion = nn.BCEWithLogitsLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

    # Train for a few epochs
    num_epochs = 3
    best_val_loss = float("inf")

    for epoch in range(num_epochs):
        train_loss, train_acc = run_epoch(model, train_loader, criterion, optimizer, training=True)
        val_loss, val_acc = run_epoch(model, val_loader, criterion, training=False)

        print(f"  Trial {trial.number}, Epoch {epoch + 1}/{num_epochs}: val_loss={val_loss:.4f}, val_acc={val_acc:.4f}")

        if val_loss < best_val_loss:
            best_val_loss = val_loss

        # Prune trial if validation loss is not improving
        trial.report(val_loss, epoch)
        if trial.should_prune():
            raise optuna.TrialPruned()

    return best_val_loss

In [11]:
# Create a study and run the hyperparameter search
study = optuna.create_study(
    direction="minimize",  # Minimize validation loss
    pruner=MedianPruner(),
)

study.optimize(objective, n_trials=6, show_progress_bar=True)

[I 2026-07-01 23:11:29,941] A new study created in memory with name: no-name-664f6516-65cf-4fc6-af20-eb1c1e9e4569
  0%|          | 0/6 [00:00<?, ?it/s]

  Trial 0, Epoch 1/3: val_loss=0.6446, val_acc=0.6329
  Trial 0, Epoch 2/3: val_loss=0.6248, val_acc=0.6578


Best trial: 0. Best value: 0.62478:  17%|█▋        | 1/6 [01:31<07:38, 91.72s/it]

  Trial 0, Epoch 3/3: val_loss=0.6284, val_acc=0.6587
[I 2026-07-01 23:13:01,659] Trial 0 finished with value: 0.6247803504233593 and parameters: {'learning_rate': 0.00011390519090289999, 'embedding_dim': 64, 'hidden_dim': 256, 'num_layers': 1, 'dropout': 0.4651430319999955, 'batch_size': 16}. Best is trial 0 with value: 0.6247803504233593.
  Trial 1, Epoch 1/3: val_loss=0.6648, val_acc=0.6027
  Trial 1, Epoch 2/3: val_loss=0.6388, val_acc=0.6311


Best trial: 0. Best value: 0.62478:  33%|███▎      | 2/6 [02:59<05:58, 89.65s/it]

...

  Trial 3, Epoch 3/3: val_loss=0.6595, val_acc=0.6184
[I 2026-07-01 23:18:34,262] Trial 3 finished with value: 0.6594527768088387 and parameters: {'learning_rate': 0.000782089346946777, 'embedding_dim': 256, 'hidden_dim': 512, 'num_layers': 3, 'dropout': 0.22424801007169243, 'batch_size': 64}. Best is trial 2 with value: 0.5831753521949261.
  Trial 4, Epoch 1/3: val_loss=0.7090, val_acc=0.5074
  Trial 4, Epoch 2/3: val_loss=0.6918, val_acc=0.5297


Best trial: 2. Best value: 0.583175:  83%|████████▎ | 5/6 [08:41<01:44, 104.69s/it]

  Trial 4, Epoch 3/3: val_loss=0.7030, val_acc=0.5157
[I 2026-07-01 23:20:11,007] Trial 4 finished with value: 0.6917595734129419 and parameters: {'learning_rate': 0.002184389541726629, 'embedding_dim': 64, 'hidden_dim': 512, 'num_layers': 3, 'dropout': 0.3284073096670489, 'batch_size': 64}. Best is trial 2 with value: 0.5831753521949261.


Best trial: 2. Best value: 0.583175: 100%|██████████| 6/6 [09:16<00:00, 92.71s/it] 

  Trial 5, Epoch 1/3: val_loss=0.6929, val_acc=0.5372
[I 2026-07-01 23:20:46,230] Trial 5 pruned. 


In [12]:
# Print best trial results
best_trial = study.best_trial

print(f"Best validation loss: {best_trial.value:.4f}")
print("Best hyperparameters:")
for key, value in best_trial.params.items():
    print(f"  {key}: {value}")

Best validation loss: 0.5832
Best hyperparameters:
  learning_rate: 0.0007658966982641726
  embedding_dim: 256
  hidden_dim: 128
  num_layers: 2
  dropout: 0.2410770033429619
  batch_size: 16


## Final Training with Best Hyperparameters

Train the final model using the best hyperparameters found by Optuna on the full train set, and evaluate on the test set.

In [13]:
# Extract best hyperparameters
best_params = best_trial.params
best_learning_rate = best_params["learning_rate"]
best_embedding_dim = best_params["embedding_dim"]
best_hidden_dim = best_params["hidden_dim"]
best_num_layers = best_params["num_layers"]
best_dropout = best_params["dropout"]
best_batch_size = best_params["batch_size"]

# Create data loaders with best batch size
train_loader = DataLoader(LyricsDataset(train_df), batch_size=best_batch_size, shuffle=True)
val_loader = DataLoader(LyricsDataset(val_df), batch_size=best_batch_size, shuffle=False)
test_loader = DataLoader(LyricsDataset(test_df), batch_size=best_batch_size, shuffle=False)

# Initialize final model with best hyperparameters
model = RNNClassifier(
    vocab_size=len(tokenizer),
    embedding_dim=best_embedding_dim,
    hidden_dim=best_hidden_dim,
    num_layers=best_num_layers,
    dropout=best_dropout,
).to(device)

criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=best_learning_rate)

print("Final model architecture:")
print(model)

Final model architecture:
RNNClassifier(
  (embedding): Embedding(30522, 256, padding_idx=0)
  (rnn): RNN(256, 128, num_layers=2, batch_first=True, dropout=0.2410770033429619, bidirectional=True)
  (dropout): Dropout(p=0.2410770033429619, inplace=False)
  (fc): Linear(in_features=256, out_features=1, bias=True)
)


In [14]:
num_epochs = 10
best_val_loss = float("inf")
best_model_path = Path("rnn_lyrics_classifier.pt")
history = []

for epoch in range(1, num_epochs + 1):
    train_loss, train_acc = run_epoch(model, train_loader, criterion, optimizer, training=True)
    val_loss, val_acc = run_epoch(model, val_loader, criterion, training=False)

    history.append(
        {
            "epoch": epoch,
            "train_loss": train_loss,
            "train_acc": train_acc,
            "val_loss": val_loss,
            "val_acc": val_acc,
        }
    )

    print(
        f"Epoch {epoch:02d} | "
        f"train_loss={train_loss:.4f} train_acc={train_acc:.4f} | "
        f"val_loss={val_loss:.4f} val_acc={val_acc:.4f}"
    )

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), best_model_path)

history_df = pd.DataFrame(history)
history_df

Epoch 01 | train_loss=0.6637 train_acc=0.5975 | val_loss=0.6602 val_acc=0.6053
Epoch 02 | train_loss=0.6370 train_acc=0.6393 | val_loss=0.6708 val_acc=0.6001
Epoch 03 | train_loss=0.6033 train_acc=0.6700 | val_loss=0.6388 val_acc=0.6381
Epoch 04 | train_loss=0.5822 train_acc=0.6958 | val_loss=0.6404 val_acc=0.6368
Epoch 05 | train_loss=0.5576 train_acc=0.7163 | val_loss=0.5884 val_acc=0.6984
Epoch 06 | train_loss=0.5792 train_acc=0.6911 | val_loss=0.6256 val_acc=0.6687
Epoch 07 | train_loss=0.5327 train_acc=0.7331 | val_loss=0.5833 val_acc=0.7155
Epoch 08 | train_loss=0.4796 train_acc=0.7694 | val_loss=0.5629 val_acc=0.7395
Epoch 09 | train_loss=0.4372 train_acc=0.7996 | val_loss=0.5709 val_acc=0.7456
Epoch 10 | train_loss=0.4141 train_acc=0.8147 | val_loss=0.5700 val_acc=0.7264


,epoch,train_loss,train_acc,val_loss,val_acc
0,1,0.663728,0.597530,0.660246,0.605332
1,2,0.636992,0.639329,0.670752,0.600087
2,3,0.603336,0.670036,0.638804,0.638112
3,4,0.582237,0.695771,0.640415,0.636801
4,5,0.557629,0.716315,0.588430,0.698427
5,6,0.579206,0.691127,0.625604,0.668706
6,7,0.532725,0.733144,0.583346,0.715472
7,8,0.479553,0.769369,0.562870,0.739510
8,9,0.437246,0.799585,0.570894,0.745629
9,10,0.414078,0.814720,0.569995,0.726399


In [15]:
# Load best checkpoint and evaluate on test
model.load_state_dict(torch.load(best_model_path, map_location=device))
test_loss, test_acc = run_epoch(model, test_loader, criterion, training=False)

print(f"\nTest loss: {test_loss:.4f}")
print(f"Test accuracy: {test_acc:.4f}")


Test loss: 0.5789
Test accuracy: 0.7168


In [16]:
artifact_dir = Path("rnn_artifacts")
artifact_dir.mkdir(exist_ok=True)

model_path = artifact_dir / "lyrics_rnn.pt"
vocab_path = artifact_dir / "lyrics_vocab.json"
params_path = artifact_dir / "best_hyperparams.json"

torch.save(model.state_dict(), model_path)
with open(vocab_path, "w", encoding="utf-8") as vocab_file:
    json.dump(vocab, vocab_file, ensure_ascii=False, indent=2)
with open(params_path, "w", encoding="utf-8") as params_file:
    json.dump(best_params, params_file, indent=2)

print("Artifacts saved:")
print(f"  Model: {model_path}")
print(f"  Vocab: {vocab_path}")
print(f"  Best hyperparameters: {params_path}")

Artifacts saved:
  Model: rnn_artifacts\lyrics_rnn.pt
  Vocab: rnn_artifacts\lyrics_vocab.json
  Best hyperparameters: rnn_artifacts\best_hyperparams.json


In [17]:
def predict_text(text: str):
    model.eval()
    cleaned_text = clean_text(text)
    input_ids, length = numericalize(cleaned_text)
    with torch.no_grad():
        logits = model(input_ids.unsqueeze(0).to(device), length.unsqueeze(0).to(device))
        probability = torch.sigmoid(logits).item()
        prediction = int(probability >= 0.5)
    return probability, prediction


sample_probability, sample_prediction = predict_text("you in the dark")
print({"probability": sample_probability, "prediction": sample_prediction})

{'probability': 0.15112681686878204, 'prediction': 0}
